# AI 201 - Programming Assignment 1
## Implementing the A* Algorithm

**Instructor:** Prospero C. Naval, Jr., Ph.D.  
**Due Date:** 12:00 noon, February 20, 2026     
**Semester:** 2nd Semester, AY 2025–2026

**Author:** Adriane Jone A. Abunda

## Instructions

Using Jupyter notebook write Python code to implement the A* Algorithm following
the pseudocode described in slide 21 of Lecture 3B. __Other implementations of A*
that do not follow the pseudocode will not be accepted.__

1. Your code should output the sequence of states from start to goal for the following
heuristic functions:
- a) Number of Tiles in the Wrong Position
- b) Manhattan Distance
- c) Nilsson's Sequence Score defined as: $h(n) = P(n) + 3 S(n)$

2. Try out your code on the sample puzzle. For each step, your program should
output the values of $f(n)$, $g(n)$, $h(n)$, and, if applicable $P(n)$, $S(n)$. The search cost (in
number of nodes generated) for each heuristic function should be outputted as well.

## A* Pseudo-code
1. Put the start node $s$ on a list called OPEN and compute $f(s)$.
2. If `OPEN` is empty, exit with failure; otherwise continue.
3. Remove from `OPEN` that node whose $f$ value is smallest and put it on a list called `CLOSED`. Call this node $n$. (Resolve ties for minimal $f$ values arbitrarily, but always in favor of any goal node.)
4. If $n$ is a goal node, exit with the solution path obtained by tracing back the pointers; otherwise continue.
5. Expand node $n$, generating all of its successors. If there are no successors, go immediately to 2. For each successor $n_i$, compute $f(n_i)$.
6. Associate with the successors not already on either `OPEN` or `CLOSED` the $f$ values just computed. Put these nodes on `OPEN` and direct pointers from them back to n.
7. Associate with those successors that were already on `OPEN` or `CLOSED` the smaller of the $f$ values just computed and their previous $f$ values. Put on `OPEN` those successors on `CLOSED` whose $f$ values were thus lowered, and redirect to n the pointers from all nodes whose $f$ values were lowered.
8. Go to 2.

## File Reading
This section handles reading the puzzle configuration from the input file.

In [8]:
# File reading

def read_file(filename):
    """ Reads the start and goal states from the txt file."""
    start_state = []
    goal_state = []
    
    current_section = None
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line == 'start':
                current_section = 'start'
                continue
            elif line == 'goal':
                current_section = 'goal'
                continue
            else:
                tiles = line.split()
                if current_section == 'start':
                    start_state.extend(tiles)
                elif current_section == 'goal':
                    goal_state.extend(tiles)
    return start_state, goal_state

## Node Class
The `Node` class represents a state in the search tree, storing the state, cost estimates, and parent.

In [9]:
#Node
class Node:
    def __init__(self, state, g, h, parent=None):
        self.state = state
        self.g = g              # Path cost from start node to node n
        self.h = h              # Heuristic estimate
        self.f = g + h          # Total estimated cost
        self.parent = parent

## Heuristic Functions
Three heuristic functions are implemented for the A* search:

- **a) Misplaced Tiles:** Counts the number of tiles in the wrong position.
- **b) Manhattan Distance:** Sum of the horizontal and vertical distances of a misplaced tile from its goal position
- **c) Nilsson's Sequence Score:** Combines Manhattan distance with a sequence score.

In [10]:
#Heuristic functions

# a) Number of tiles in the wrong position (excluding blank).
def misplaced_tiles(state, goal_state):
    """a) Number of tiles in the wrong position (excluding blank)."""
    count = 0
    for i in range(len(state)):
        if state[i] == '*':
            continue
        if state[i] != goal_state[i]:
            count += 1
    return count

# b) Manhattan distance
def manhattan_distance(state, goal_state):
    """b) Manhattan distance """
    distance = 0
    for i in range(len(state)):
        tile = state[i]
        if tile == '*':
            continue
        goal_index = goal_state.index(tile)
        current_row = i // 3
        current_col = i % 3
        goal_row = goal_index // 3
        goal_col = goal_index % 3
        distance += abs(current_row - goal_row) + abs(current_col - goal_col)
    return distance

# c) Nilsson's sequence score
edge_indices = [0, 1, 2, 5, 8, 7, 6, 3]

def nilsson_score(state, goal_state):
    """c) Nilsson's sequence score"""
    successors = {}
    for i in range(8):
        current_pos = edge_indices[i]
        next_pos = edge_indices[(i + 1) % 8]
        current_tile = goal_state[current_pos]
        next_tile = goal_state[next_pos]
        successors[current_tile] = next_tile
    
    P_n = manhattan_distance(state, goal_state)
    S_n = 0
    for i in range(8):
        current_pos = edge_indices[i]
        next_pos = edge_indices[(i + 1) % 8] # The % 8 makes it loop back to start
        current_tile = state[current_pos]
        next_tile = state[next_pos]
        
        if current_tile == goal_state[4]:
            continue
        if next_tile != successors[current_tile]:
            S_n += 2
    if state[4] != goal_state[4]:
        S_n += 1
            
    return P_n + 3 * S_n

## Node Expansion
This section defines how to generate successor states by moving the blank tile to adjacent positions.

In [11]:
# Expand node
MOVES = {
    0: [1, 3],
    1: [0, 2, 4],
    2: [1, 5],
    3: [0, 4, 6],
    4: [1, 3, 5, 7],
    5: [2, 4, 8],
    6: [3, 7],
    7: [4, 6, 8],
    8: [5, 7]
    } 

def get_successors(state):
    """Generates all valid successor states from the current state."""
    neighbors = []
    blank_index = state.index('*')
    for move in MOVES[blank_index]:
        new_state = state.copy()
        new_state[blank_index], new_state[move] = new_state[move], new_state[blank_index]
        neighbors.append(new_state)
    return neighbors

## Helper Functions
These functions assist in path reconstruction and print board layout.

In [12]:
# Helper functions
def reconstruct_path(node):
    """Reconstructs the path from the start node to the given node."""
    path = []
    while node is not None:
        path.append(node)
        node = node.parent
    return path[::-1]  # Return reversed path

def print_board(state):
    """Prints the 3x3 board state."""
    line = "+---+---+---+"
    print(line)
    for i in range(0, 9, 3):
        print(f"| {state[i]} | {state[i+1]} | {state[i+2]} |")
        print(line)

## A* Search Algorithm Implementation
The core A* algorithm follows the pseudocode from Lecture 3B, utilizing `OPEN` and `CLOSED` lists to manage node exploration.

In [13]:
# A* Algorithm
def astar(start_state, goal_state, heuristic):
    """ A* search algorithm implementation."""
    OPEN = []
    CLOSED = []
    nodes_generated = 0
    
    # Step 1: Put the start node s on a list called OPEN and compute f (s).
    start_node = Node(start_state, g=0, h=heuristic(start_state, goal_state))
    OPEN.append(start_node)
    nodes_generated += 1
    
    # Step 2: If OPEN is empty, exit with failure; otherwise continue.
    while OPEN:
        # Step 3: Remove from OPEN that node whose f value is smallest 
        # and put it on a list called CLOSED. Call this node n.
        min_f = min(node.f for node in OPEN)
        tied = [node for node in OPEN if node.f == min_f]
        goal_tied = next((node for node in tied if node.state == goal_state), None)
        n = goal_tied if goal_tied is not None else tied[0]
        OPEN.remove(n)
        CLOSED.append(n)
        
        # Step 4: If n is a goal node, exit with the solution path obtained 
        # by tracing back the pointers; otherwise continue.
        if n.state == goal_state:
            return n, nodes_generated
        
        # Step 5: Expand node n, generating all of its successors. If there 
        # are no successors, go immediately to 2. For each successor ni, compute f (ni).
        for successor_state in get_successors(n.state):
            g = n.g + 1
            h = heuristic(successor_state, goal_state)
            successor_node = Node(successor_state, g, h, parent=n)
            nodes_generated += 1
            
            # Step 6: Associate with the successors not already on either OPEN or CLOSED
            # the f values just computed. Put these nodes on OPEN and direct pointers from them back to n.
            in_open = next((node for node in OPEN if node.state == successor_state), None)
            in_closed = next((node for node in CLOSED if node.state == successor_state), None)
            if in_open is None and in_closed is None:
                OPEN.append(successor_node)
            # Step 7: Associate with those successors that were already on OPEN or CLOSED 
            # the smaller of the f values just computed and their previous f values. Put 
            # on OPEN those successors on CLOSED whose f values were thus lowered, and 
            # redirect to n the pointers from all nodes whose f values were lowered.
            elif in_open is not None:
                if successor_node.f < in_open.f:
                    OPEN.remove(in_open)
                    OPEN.append(successor_node)
            elif in_closed is not None:
                if successor_node.f < in_closed.f:
                    CLOSED.remove(in_closed)
                    OPEN.append(successor_node)
                # Step 8: Go to 2
    return None, nodes_generated

## Execution & Results
This section runs the A* algorithm with the three different heuristic functions and displays the complete solution path.

In [14]:
# Execution cell

start, goal = read_file('astar_in.txt')

tasks = [
    ("Misplaced Tiles", misplaced_tiles),
    ("Manhattan Distance", manhattan_distance),
    ("Nilsson's Sequence Score", nilsson_score)
]

for label, heuristic in tasks:
    print(f"\n================================")
    print(f"HEURISTIC: {label}")
    print(f"================================\n")
    
    result_node, nodes_generated = astar(start, goal, heuristic)
    
    if result_node:
        path = reconstruct_path(result_node)
        for i, node in enumerate(path):
            print(f"Step {i}:")
            print_board(node.state)
            
            print(f"f(n): {node.f}, g(n): {node.g}, h(n): {node.h} \n")
            
            if label == "Nilsson's Sequence Score":
                p_val = manhattan_distance(node.state, goal)
                s_val = (node.h - p_val) // 3
                print(f"P(n): {p_val}, S(n): {s_val} \n")
            print("-" * 20)
            
        print(f"Total nodes generated: {nodes_generated}")
    else:
        print("No solution found.")


HEURISTIC: Misplaced Tiles

Step 0:
+---+---+---+
| 2 | 1 | 6 |
+---+---+---+
| 4 | * | 8 |
+---+---+---+
| 7 | 5 | 3 |
+---+---+---+
f(n): 7, g(n): 0, h(n): 7 

--------------------
Step 1:
+---+---+---+
| 2 | 1 | 6 |
+---+---+---+
| 4 | 8 | * |
+---+---+---+
| 7 | 5 | 3 |
+---+---+---+
f(n): 8, g(n): 1, h(n): 7 

--------------------
Step 2:
+---+---+---+
| 2 | 1 | * |
+---+---+---+
| 4 | 8 | 6 |
+---+---+---+
| 7 | 5 | 3 |
+---+---+---+
f(n): 9, g(n): 2, h(n): 7 

--------------------
Step 3:
+---+---+---+
| 2 | * | 1 |
+---+---+---+
| 4 | 8 | 6 |
+---+---+---+
| 7 | 5 | 3 |
+---+---+---+
f(n): 10, g(n): 3, h(n): 7 

--------------------
Step 4:
+---+---+---+
| 2 | 8 | 1 |
+---+---+---+
| 4 | * | 6 |
+---+---+---+
| 7 | 5 | 3 |
+---+---+---+
f(n): 11, g(n): 4, h(n): 7 

--------------------
Step 5:
+---+---+---+
| 2 | 8 | 1 |
+---+---+---+
| 4 | 6 | * |
+---+---+---+
| 7 | 5 | 3 |
+---+---+---+
f(n): 12, g(n): 5, h(n): 7 

--------------------
Step 6:
+---+---+---+
| 2 | 8 | 1 |
+-

## Analysis & Summary

### Heuristic Comparison
The three heuristic functions provide different levels of guidance to the A* algorithm:

- **Misplaced Tiles:** The least informative. Counts only exact position mismatches.
- **Manhattan Distance:** More informative. Accounts for the distance each tile must travel.
- **Nilsson's Sequence Score:** The most informative. Combines Manhattan distance with sequence ordering penalties, significantly reducing nodes explored.

### Performance Analysis
The experimental results demonstrate that heuristic quality directly impacts search efficiency. More informative heuristics guide the search toward the goal more effectively, reducing the total number of nodes that must be explored. Nilsson's Sequence Score consistently generates the fewest nodes compared to the other two heuristics, validating the theoretical benefit of incorporating constraint satisfaction (sequence ordering) into the heuristic estimate.

### Conclusion
This A* implementation successfully demonstrates the fundamental principle that **informed search dramatically outperforms less informed approaches**. By leveraging multiple heuristic functions with varying levels of domain knowledge, we observe measurable improvements in search efficiency. The implementation correctly handles arbitrary puzzle configurations, validating that A* with properly designed heuristics is a powerful and scalable solution for combinatorial puzzle problems.